In [ ]:

import itertools
import random
from collections import defaultdict

class AgentJuan:
    PRIMER_INTENTO = "3579"

    # key=(fijas,picas) ; value=(segunda_jugada, peor_caso_turno2, restantes_turno1)
    TABLA_G2 = {
        (0, 0): ("0124", 88, 360),
        (0, 1): ("0123", 378, 1440),
        (0, 2): ("0135", 304, 1260),
        (0, 3): ("0795", 75, 264),
        (0, 4): ("5397", 4, 9),

        (1, 0): ("0129", 126, 480),
        (1, 1): ("0139", 180, 720),
        (1, 2): ("0359", 80, 216),
        (1, 3): ("3795", 4, 8),

        (2, 0): ("0179", 48, 180),
        (2, 1): ("0379", 30, 72),
        (2, 2): ("3597", 4, 6),

        (3, 0): ("0579", 15, 24),

        (4, 0): ("3579", 1, 1),
    }

    def __init__(self):
        # Solo para que el print del environment no sea None (como pediste)
        self.secret = "7854"

        # Universo de números válidos (4 dígitos sin repetir)
        self._all = [''.join(map(str, p)) for p in itertools.permutations(range(10), 4)]

        # --- Adivinador (modelo del secreto del rival) ---
        self._cands_other = self._all[:]     # candidatos del rival
        self._guess_turn = 1
        self._last_guess = self.PRIMER_INTENTO
        self._turn1_percept = None           # (picas,fijas) recibido al primer guess

        # --- Villano (modelo para responder sobre "mi secreto") ---
        self._cands_me = self._all[:]        # candidatos consistentes con mis respuestas
        self._respond_turn = 1

    # -----------------------------
    # Helpers internos
    # -----------------------------
    def _is_valid_guess(self, g: str) -> bool:
        return isinstance(g, str) and len(g) == 4 and g.isdigit() and len(set(g)) == 4

    def _score(self, guess: str, secret: str):
        """Devuelve (picas,fijas)."""
        fijas = sum(a == b for a, b in zip(guess, secret))
        coinc = sum(d in secret for d in guess)
        picas = coinc - fijas
        return (picas, fijas)

    def _filter_by_feedback(self, cands, guess, result_pf):
        return [s for s in cands if self._score(guess, s) == result_pf]

    def _partition_by_feedback(self, cands, guess):
        groups = defaultdict(list)
        for s in cands:
            groups[self._score(guess, s)].append(s)
        return groups

    def _minimax_guess(self, cands):
        """Elige guess minimizando el peor caso (máximo grupo)."""
        best = None
        best_worst = float('inf')

        for g in cands:  # simple: buscar dentro de candidatos actuales
            counts = defaultdict(int)
            for s in cands:
                counts[self._score(g, s)] += 1
            worst = max(counts.values())
            if worst < best_worst:
                best_worst = worst
                best = g

        return best

    # -----------------------------
    # API del Environment
    # -----------------------------
    def compute(self, percepts):
        """
        percepts: None o (picas,fijas) que el rival devolvió a mi guess anterior.
        Retorna siguiente guess para adivinar el secreto del rival.
        """
        if percepts is not None:
            self._cands_other = self._filter_by_feedback(self._cands_other, self._last_guess, percepts)

            # fallback defensivo si el rival respondió inconsistente
            if not self._cands_other:
                self._cands_other = self._all[:]

            if self._guess_turn == 1:
                self._turn1_percept = percepts

            self._guess_turn += 1

        if self._guess_turn == 1:
            self._last_guess = self.PRIMER_INTENTO
            return self._last_guess

        if self._guess_turn == 2:
            # percept está como (picas,fijas); la tabla usa (fijas,picas)
            if self._turn1_percept is not None:
                picas, fijas = self._turn1_percept
                key = (fijas, picas)
                if key in self.TABLA_G2:
                    self._last_guess = self.TABLA_G2[key][0]
                    return self._last_guess

            self._last_guess = self._cands_other[0]
            return self._last_guess

        self._last_guess = self._minimax_guess(self._cands_other)
        return self._last_guess

    def respond(self, guess):
        """
        Villano real (no usa secreto):
          - primer respond: SIEMPRE (picas,fijas) = (1,0)
          - luego: elige feedback que deja MÁS candidatos consistentes
        """
        if not self._is_valid_guess(guess):
            return (0, 0)

        # 1) Primer respond fijo: 1 pica, 0 fijas
        if self._respond_turn == 1:
            self._respond_turn += 1
            # Filtrar candidatos para mantener consistencia con lo que acabo de decir
            self._cands_me = self._filter_by_feedback(self._cands_me, guess, (1, 0))
            # Si por alguna razón quedara vacío (no debería con reglas normales), caemos al peor caso real
            if not self._cands_me:
                self._cands_me = self._all[:]
                groups = self._partition_by_feedback(self._cands_me, guess)
                chosen = max(groups.items(), key=lambda kv: len(kv[1]))[0]
                self._cands_me = groups[chosen]
                return chosen
            return (1, 0)

        # 2) Turnos siguientes: peor caso consistente
        groups = self._partition_by_feedback(self._cands_me, guess)
        chosen = max(groups.items(), key=lambda kv: len(kv[1]))[0]
        self._cands_me = groups[chosen]
        self._respond_turn += 1
        return chosen


In [ ]:
from itertools import permutations
import random

# Corporacion Cruceta
# Omar Chaparro
# Cristian Arcia

class Cruceta:
    def __init__(self):
        self.digits = "01234567890"
        self.space = [''.join(p) for p in permutations(self.digits, 4)]
        self.secret = "".join(random.sample(self.digits, 4))
        self.last_guess = None


    def compare(self, guess, target):
        picas = fijas = 0
        for d,s in zip(guess, target):
            if d == s:
                fijas += 1
            elif d in target:
                picas += 1

        return (picas, fijas)

    def respond(self, guess):
        return self.compare(guess, self.secret)

    def compute(self, percep):
        if percep != None:
            new_space = [x for x in self.space if (percep == self.compare(x, self.last_guess))]
            self.space = new_space
        self.last_guess = self.space[0]
        return self.last_guess

In [ ]:
import random
import itertools
from collections import defaultdict

class Crucetero:
    def __init__(self):
        self.all_codes = list(itertools.permutations(range(10), 4))
        self.candidates = self.all_codes.copy()

        self.secret = random.choice(self.all_codes)

        self.last_guess = None

    def compute(self, percept):
        if percept is not None and self.last_guess is not None:
            picas, fijas = percept
            self._filter_candidates(self.last_guess, picas, fijas)
        if self.last_guess is None:
            guess = (0, 1, 2, 3)
        else:
            guess = self._minimax_guess()

        self.last_guess = guess
        return ''.join(map(str, guess))

    def respond(self, guess):
        guess_tuple = tuple(int(d) for d in guess)
        fijas = sum(g == s for g, s in zip(guess_tuple, self.secret))
        picas = sum(min(guess_tuple.count(d), self.secret.count(d)) for d in set(guess_tuple)) - fijas
        return (picas, fijas)

    def _filter_candidates(self, guess, picas, fijas):
        new_candidates = []
        for code in self.candidates:
            f, p = self._score(guess, code)
            if f == fijas and p == picas:
                new_candidates.append(code)
        self.candidates = new_candidates

    def _score(self, guess, code):
        fijas = sum(g == c for g, c in zip(guess, code))
        picas = sum(min(guess.count(d), code.count(d)) for d in set(guess)) - fijas
        return fijas, picas

    def _minimax_guess(self):
        best_guess = None
        best_worst_case = float('inf')

        possible_guesses = self.all_codes

        for guess in possible_guesses:
            partitions = defaultdict(int)

            for code in self.candidates:
                score = self._score(guess, code)
                partitions[score] += 1

            worst_case = max(partitions.values())

            if worst_case < best_worst_case:
                best_worst_case = worst_case
                best_guess = guess

            if worst_case == best_worst_case and guess in self.candidates:
                best_guess = guess
                break

        return best_guess

In [ ]:
import math
import random
import itertoolse
from collections import defaultdict

class CrucetaEntropica:
    def __init__(self):
        self.all_codes = list(itertools.permutations(range(10), 4))
        self.candidates = self.all_codes.copy()

        self.secret = random.choice(self.all_codes)

        self.last_guess = None

    def compute(self, percept):
        if percept is not None and self.last_guess is not None:
            picas, fijas = percept
            self._filter_candidates(self.last_guess, picas, fijas)
        if self.last_guess is None:
            guess = (0, 1, 2, 3)
        else:
            guess = self._entropy_guess()

        self.last_guess = guess
        return ''.join(map(str, guess))

    def respond(self, guess):
        guess_tuple = tuple(int(d) for d in guess)
        fijas = sum(g == s for g, s in zip(guess_tuple, self.secret))
        picas = sum(min(guess_tuple.count(d), self.secret.count(d)) for d in set(guess_tuple)) - fijas
        return (picas, fijas)

    def _filter_candidates(self, guess, picas, fijas):
        new_candidates = []
        for code in self.candidates:
            f, p = self._score(guess, code)
            if f == fijas and p == picas:
                new_candidates.append(code)
        self.candidates = new_candidates

    def _score(self, guess, code):
        fijas = sum(g == c for g, c in zip(guess, code))
        picas = sum(min(guess.count(d), code.count(d)) for d in set(guess)) - fijas
        return fijas, picas

    def _entropy_guess(self):
        best_guess = None
        best_entropy = -1

        if len(self.candidates) <= 100:
            possible_guesses = self.candidates
        else:
            possible_guesses = self.all_codes

        for guess in possible_guesses:
            partitions = defaultdict(int)

            for code in self.candidates:
                score = self._score(guess, code)
                partitions[score] += 1

            total = len(self.candidates)
            entropy = 0.0

            for count in partitions.values():
                p = count / total
                entropy -= p * math.log2(p)

            if entropy > best_entropy:
                best_entropy = entropy
                best_guess = guess

        return best_guess

In [ ]:
import itertools
import random

class AgentRandom:
    def __init__(self):
        self.all_codes = list(itertools.permutations(range(10), 4))
        self.secret = random.choice(self.all_codes)

    def compute(self, percept):
        guess = random.choice(self.all_codes)
        return ''.join(map(str, guess))

    def respond(self, guess):
        guess_tuple = tuple(int(d) for d in guess)
        fijas = sum(g == s for g, s in zip(guess_tuple, self.secret))
        picas = sum(min(guess_tuple.count(d), self.secret.count(d)) for d in set(guess_tuple)) - fijas
        return (picas, fijas)

In [ ]:
from itertools import permutations
import random

class AgentA:
    def __init__(self):
        self.digits = "01234567890"
        self.space = [''.join(p) for p in permutations(self.digits, 4)]
        self.secret = "".join(random.sample(self.digits, 4))
        self.last_guess = None


    def compare(self, guess, target):
        picas = fijas = 0
        for d,s in zip(guess, target):
            if d == s:
                fijas += 1
            elif d in target:
                picas += 1

        return (picas, fijas)

    def respond(self, guess):
        return self.compare(guess, self.secret)

    def compute(self, percep):
        if percep != None:
            new_space = [x for x in self.space if (percep == self.compare(x, self.last_guess))]
            self.space = new_space
        self.last_guess = self.space[0]
        return self.last_guess

In [ ]:
import itertools
import random
from collections import defaultdict

class Environment:
    def __init__(self, agentA, agentB):
        self.agentA = agentA
        self.agentB = agentB

    def send(self, sender, guess):
        if sender == 'A':
            return self.agentB.respond(guess)
        else:
            return self.agentA.respond(guess)

A = Cruceta()
B = AgentJuan()
env = Environment(A, B)

agents = [
    ('A', A),
    ('B', B)
]

random.shuffle(agents)

percepts = {'A': None, 'B': None}

print(f'Secret A: {A.secret}')
print(f'Secret B: {B.secret}')
print(f'Empieza el Agente {agents[0][0]}')

game_over = False

for turn in range(1, 8):
    if game_over:
        break

    print(f'\n--- Turno {turn} ---')

    for name, agent in agents:
        guess = agent.compute(percepts[name])
        result = env.send(name, guess)
        percepts[name] = result

        print(f'Agente {name} → {guess} | Picas/Fijas: {result}')

        if result[1] == 4:
            print(f'\n Agente {name} ganó')
            game_over = True
            break

if not game_over:
    print('Nadie ganó')

Secret A: 0928
Secret B: 7854
Empieza el Agente A

--- Turno 1 ---
Agente A → 0123 | Picas/Fijas: (1, 0)
Agente B → 3579 | Picas/Fijas: (1, 0)

--- Turno 2 ---
Agente A → 1456 | Picas/Fijas: (1, 0)
Agente B → 0123 | Picas/Fijas: (0, 2)

--- Turno 3 ---
Agente A → 2578 | Picas/Fijas: (2, 0)
Agente B → 0147 | Picas/Fijas: (0, 1)

--- Turno 4 ---
Agente A → 3687 | Picas/Fijas: (2, 0)
Agente B → 0625 | Picas/Fijas: (0, 2)

--- Turno 5 ---
Agente A → 5739 | Picas/Fijas: (1, 1)
Agente B → 0683 | Picas/Fijas: (1, 1)

--- Turno 6 ---
Agente A → 6792 | Picas/Fijas: (0, 4)

 Agente A ganó
